# Chapter 21 — A bounded adaptive-control sandbox

Build a complete feedback loop using a dimensionless toy process, then explain every action. This is a model-based engineering exercise, not a physiological model or stimulation prescription. It needs a CPU and Python 3.11+. No human recordings, headset, API key, or device are used. All outputs are intentionally empty until you run the notebook.

Read SIMULATION_CONTRACT.md and ASSIGNMENT.md beside the notebook. Work through the chapter's control-direction and delay examples before interpreting an aggregate result.

## 1. Load the inspectable helper

Run locally from this folder, or open the notebook in Colab. The bootstrap fetches an immutable helper revision only if the local file is absent and verifies its SHA-256. Inspect the source. The digest checks content against this notebook's expected value; it is not operator authentication. Keep this baseline intact when developing a separately versioned extension.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import urllib.request

HELPER_COMMIT = "f6e8f3bdaeb88b551a7217071074f0526b781a59"
EXPECTED_SHA256 = "3e3eaf24f573d3621c5ffea1a6595c6f9671f8c2feda7178b81c6f748d07936d"
helper_path = Path("adaptive_sandbox.py")
if not helper_path.exists():
    url = (
        "https://raw.githubusercontent.com/ZagElSayed/"
        "Introduction-to-Neuroinformatics/" + HELPER_COMMIT +
        "/chapters/ch21-closed-loop/adaptive_sandbox.py"
    )
    with urllib.request.urlopen(url, timeout=30) as response:
        source = response.read()
    if hashlib.sha256(source).hexdigest() != EXPECTED_SHA256:
        raise RuntimeError("Downloaded helper differs from the verified baseline.")
    helper_path.write_bytes(source)
if hashlib.sha256(helper_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Local helper differs; retain the baseline and version extensions separately.")
spec = importlib.util.spec_from_file_location("adaptive_sandbox", helper_path)
lab = importlib.util.module_from_spec(spec)
import sys
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
print(lab.VERSION)


## 2. Compare fixed and bounded adaptive control

The latent state starts at 1.0; an issued action is applied two ticks later when advancing the process. Larger positive action lowers this toy state. The controller sees only noisy observations, age, and a stop flag. Its smoother uses alpha 0.25. Normal output is bounded to [0,1] with step change at most 0.10. The adaptive rule changes gain only after 20 eligible errors, within [0.50,2.0]. Predict what you expect; the data may disagree. A shared seed gives matched exogenous noise, not identical observations after the policies diverge.

In [ ]:
from dataclasses import replace
cfg = lab.Config(seed=21)
fixed = lab.simulate(cfg, adaptive=False, faults=False)
adaptive = lab.simulate(cfg, adaptive=True, faults=False)
for name, result in [("fixed", fixed), ("adaptive", adaptive)]:
    print(name, result["summary"])


## 3. Read the trajectory

Plot issued and applied action separately. The latent state is an audit-only quantity; do not pass it to a decision rule. Target-band performance uses all ticks, including warmup, fallback, and stop. Record your interpretation before changing any parameter.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for name, color, result in [("fixed", "#167D91", fixed), ("adaptive", "#7556A8", adaptive)]:
    rows = result["rows"]
    ticks = [r["tick"] for r in rows]
    axes[0].plot(ticks, [r["audit_state"] for r in rows], label=name, color=color)
    axes[1].plot(ticks, [r["delivered"] for r in rows], label=name + " issued", color=color)
    axes[1].plot(ticks, [r["applied"] for r in rows], linestyle="--", label=name + " applied", color=color, alpha=0.6)
    axes[2].plot(ticks, [r["gain_used"] for r in rows], label=name, color=color)
axes[0].axhspan(cfg.target-cfg.target_half_width, cfg.target+cfg.target_half_width,
                color="#167D91", alpha=0.12)
axes[0].set_ylabel("Latent toy state")
axes[1].set_ylabel("Normalized action")
axes[2].set_ylabel("Gain")
axes[2].set_xlabel("Virtual tick")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 4. Challenge inhibition, stop, and recovery

Faults occur at ticks 40 (missing), 80 (age 0.75 > 0.50), 120 (measurement 5.0 exceeds magnitude 3.0), and 160–164 (manual stop). Fallback issues zero immediately and overrides normal slew limits. Previously issued actions still traverse the delay model. After any fault, three consecutive fresh observations are needed for active operation; the gain is retained, while estimator readiness and partial adaptation blocks reset. These are toy rules, not validated artifact criteria.

In [ ]:
faulted = lab.simulate(cfg, adaptive=True, faults=True)
print(faulted["summary"])
for row in faulted["rows"]:
    if row["injected"] or 165 <= row["tick"] <= 168:
        fields = ["tick", "injected", "mode", "reason", "estimate",
                  "proposed", "delivered", "applied", "gain_used", "gain_next"]
        print({key: row[key] for key in fields})


## 5. Reconstruct decisions and inspect invariants

The audit repeats decisions on the stored measurement stream, checking decision fields, action/gain bounds, inhibited outputs, and normal-operation slew. This checks a historical trajectory; it does not establish what a different policy would have done to the process. No test here certifies a device or proves biological safety.

In [ ]:
audit_result = lab.audit(faulted)
print(audit_result)
rebuilt = lab.reconstruct(faulted)
print("Reconstructed decisions:", len(rebuilt))


## 6. Development scenarios and provenance

Use seeds 21–23 for development. Keep successful and unsuccessful rollouts. The noise changes across seeds, but the default disturbance and scripted fault schedule remain the same: this evaluates noise realizations of a declared scenario, not every operating condition. Normalized action sum and variation are burden proxies, not energy or physiological dose.

In [ ]:
development = []
for seed in (21, 22, 23):
    for faults in (False, True):
        for adaptive_flag in (False, True):
            result = lab.simulate(replace(cfg, seed=seed), adaptive=adaptive_flag, faults=faults)
            lab.audit(result)
            development.append({"seed": seed, "faults": faults,
                                "adaptive": adaptive_flag, **result["summary"]})
for row in development:
    print(row)
receipt = lab.save_bundle(faulted)
print("Saved bundle receipt:", receipt)


## 7. Freeze the evaluation procedure

Before opening the final scenarios, document the policy, estimator, adaptation schedule, bounds, metrics, and anticipated limits in the system card. Set FROZEN to True only after completing that record. This is an experimental hold-out rule. Internal gain updates remain part of the frozen procedure. If you redesign after viewing final results, select genuinely new evaluation scenarios.

In [ ]:
FROZEN = False
FINAL_SEEDS = (121, 122, 123, 124, 125)


In [ ]:
if not FROZEN:
    raise RuntimeError("Document and freeze your procedure before running final scenarios.")
final_results = []
for seed in FINAL_SEEDS:
    for faults in (False, True):
        for adaptive_flag in (False, True):
            result = lab.simulate(replace(cfg, seed=seed), adaptive=adaptive_flag, faults=faults)
            lab.audit(result)
            receipt = lab.save_bundle(result)
            final_results.append({"seed": seed, "faults": faults,
                                  "adaptive": adaptive_flag,
                                  **result["summary"], "receipt": receipt})
for row in final_results:
    print(row)


## 8. Scientific interpretation

Submit all-step tracking and target-band outcomes, active coverage, normalized burden and variation, mode counts, constraints, and gain updates. Explain one normal action and one recovery trace. Compare variability across seeds without treating correlated ticks as independent participants. State whether adaptation helped the declared objective and what tradeoffs changed. Full credit does not require superiority.

Explain why this cannot establish: a human neural biomarker, an optimal stimulation dose, physical latency, device safety, or therapeutic benefit. Describe the next experiment needed for one of those claims. See the 100-point rubric and system-card template in ASSIGNMENT.md.